# Dataset Exploration: Hindi-HDTB (UD)

This notebook explores the **Hindi Head-Driven Treebank** in Universal Dependencies (UD) format.

We will load the training file, count sentences and dependency labels, and look at a few examples of labels that matter for Karaka mapping (`nsubj`, `obj`, `obl`, `iobj`, `case`).

No machine learning models are used here — only simple file reading and counting.

## About the CONLL-U Format

The file `hi_hdtb-ud-train.conllu` stores annotated Hindi sentences.

Each sentence block contains:
- **Comment lines** starting with `#` (e.g. `# text = ...` gives the full sentence)
- **Token lines** with tab-separated fields

On each token line, the important columns for us are:
- **FORM** (column 2): the word as it appears in the sentence
- **HEAD** (column 7): which word this token depends on
- **DEPREL** (column 8): the dependency label (e.g. `nsubj`, `obj`, `case`)

## 1. Load the CONLL-U File

We read the file from `data/raw/` and parse it into a list of sentences.
Each sentence is stored as a dictionary with its text and a list of tokens.

In [1]:
from collections import Counter
from pathlib import Path


def load_conllu(filepath):
    """Read a CONLL-U file and return a list of sentence dictionaries."""
    sentences = []
    current = {"text": "", "sent_id": "", "tokens": []}

    with open(filepath, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            # Blank line marks the end of a sentence
            if not line:
                if current["tokens"]:
                    sentences.append(current)
                    current = {"text": "", "sent_id": "", "tokens": []}
                continue

            # Comment lines store sentence metadata
            if line.startswith("#"):
                if line.startswith("# text = "):
                    current["text"] = line[len("# text = "):]
                elif line.startswith("# sent_id = "):
                    current["sent_id"] = line[len("# sent_id = "):]
                continue

            # Token line: skip multi-word tokens (IDs like 3-4)
            columns = line.split("\t")
            if len(columns) < 8 or "-" in columns[0]:
                continue

            current["tokens"].append({
                "id": columns[0],
                "form": columns[1],
                "lemma": columns[2],
                "head": columns[6],
                "deprel": columns[7],
            })

    # Save the last sentence if the file does not end with a blank line
    if current["tokens"]:
        sentences.append(current)

    return sentences


data_path = Path("../data/raw/hi_hdtb-ud-train.conllu")
sentences = load_conllu(data_path)

print(f"Loaded file: {data_path}")
print(f"First sentence: {sentences[0]['text']}")

Loaded file: ..\data\raw\hi_hdtb-ud-train.conllu
First sentence: यह एशिया की सबसे बड़ी मस्जिदों में से एक है ।


## 2. Count Total Sentences

Each sentence block in the file becomes one entry in our `sentences` list.

In [2]:
total_sentences = len(sentences)
print(f"Total sentences: {total_sentences}")

Total sentences: 13306


## 3. Count Unique Dependency Labels

We collect every **DEPREL** value across all tokens in the dataset.

In [3]:
all_deprels = [
    token["deprel"]
    for sentence in sentences
    for token in sentence["tokens"]
]

unique_deprels = sorted(set(all_deprels))

print(f"Total dependency relations (tokens): {len(all_deprels)}")
print(f"Unique dependency labels: {len(unique_deprels)}")
print()
print("All labels:")
print(", ".join(unique_deprels))

Total dependency relations (tokens): 281057
Unique dependency labels: 28

All labels:
acl, acl:relcl, advcl, advmod, amod, aux, aux:pass, case, cc, ccomp, compound, conj, cop, dep, det, dislocated, iobj, mark, nmod, nsubj, nsubj:pass, nummod, obj, obl, punct, root, vocative, xcomp


## 4. Most Common Dependency Labels

Some labels appear much more often than others. Here are the **20 most frequent** dependency relations in the training set.

In [4]:
deprel_counts = Counter(all_deprels)
top_20 = deprel_counts.most_common(20)

print(f"{'Label':<12} {'Count':>8}")
print("-" * 22)
for label, count in top_20:
    print(f"{label:<12} {count:>8}")

Label           Count
----------------------
case            53121
compound        31994
nmod            27266
obl             25211
punct           18668
nsubj           16926
root            13306
obj             13003
amod            11400
mark            10895
aux:pass         8451
aux              7752
det              6336
conj             5904
cc               5111
nummod           4462
dep              3926
advcl            3681
advmod           3121
cop              2735


## 5. Example Tokens by Label

These five labels are especially important for later **UD → Karaka mapping**:

| Label | Typical role |
|-------|-------------|
| `nsubj` | Nominal subject |
| `obj` | Direct object |
| `iobj` | Indirect object |
| `obl` | Oblique modifier (often with a postposition) |
| `case` | Case marker / postposition |

For each label, we print a few examples showing the word, its lemma, the label, and the word it depends on (the head).

In [5]:
labels_to_show = ["nsubj", "obj", "obl", "iobj", "case"]
examples_per_label = 3


def show_examples(sentences, label, max_examples=3):
    """Print example tokens with a given dependency label."""
    print(f"\n{'=' * 60}")
    print(f"Examples of: {label}")
    print(f"{'=' * 60}")

    found = 0
    for sentence in sentences:
        id_to_form = {token["id"]: token["form"] for token in sentence["tokens"]}

        for token in sentence["tokens"]:
            if token["deprel"] != label:
                continue

            head_form = id_to_form.get(token["head"], token["head"])
            print(f"Sentence ({sentence['sent_id']}): {sentence['text']}")
            print(
                f"  {token['form']} ({token['lemma']}) "
                f"--{label}--> {head_form}"
            )
            print()

            found += 1
            if found >= max_examples:
                return

    if found == 0:
        print(f"No examples found for label: {label}")


for label in labels_to_show:
    show_examples(sentences, label, max_examples=examples_per_label)


Examples of: nsubj
Sentence (train-s2): इसे नवाब शाहजेहन ने बनवाया था ।
  शाहजेहन (शाहजेहन) --nsubj--> बनवाया

Sentence (train-s3): इसका प्रवेश द्वार दो मंजिला है ।
  द्वार (द्वार) --nsubj--> मंजिला

Sentence (train-s4): जिसमें चार मेहराबें हैं और मुख्य प्रार्थना हॉल में जाने के लिए 9 प्रवेश द्वार हैं ।
  द्वार (द्वार) --nsubj--> हैं


Examples of: obj
Sentence (train-s2): इसे नवाब शाहजेहन ने बनवाया था ।
  इसे (यह) --obj--> बनवाया

Sentence (train-s6): यहाँ लगने वाला तीन दिन का इज्तिमा पूरे देश के लोगों को आमंत्रित करता है ।
  लोगों (लोग) --obj--> करता

Sentence (train-s7): शौकत महल के सामने बड़ी झील के किनारे स्थित वास्तुकला का यह खूबसूरत नमूना कुदसिया बेगम के काल का है जिन्हें गोहर बेगम भी कहा जाता था ।
  जिन्हें (जो) --obj--> कहा


Examples of: obl
Sentence (train-s4): जिसमें चार मेहराबें हैं और मुख्य प्रार्थना हॉल में जाने के लिए 9 प्रवेश द्वार हैं ।
  हॉल (हॉल) --obl--> हैं

Sentence (train-s6): यहाँ लगने वाला तीन दिन का इज्तिमा पूरे देश के लोगों को आमंत्रित करता है ।
  यहाँ (यहा

## Summary

In this notebook we:
1. Loaded the Hindi-HDTB training file in CONLL-U format
2. Counted the total number of sentences
3. Listed all unique dependency labels
4. Identified the 20 most common labels
5. Inspected example tokens for key labels used in Karaka mapping

These observations will inform the UD → Karaka mapping work in the next phase of the project.